In [1]:
# dataset_info = {
#     'Lipophilicity': '0205',
#     'esol': '1225', 
#     'Mutagenicity': '0205',
#     'hERG': '0205',
#     'BBBP': '0205',
#     'tox21':'0205',
# }# Choose from 'Mutagenicity', 'hERG', 'BBBP', 'clintox'
import CONSTANTS

In [3]:
import pandas as pd
from DataLoader import get_setup_files_with_folds
import numpy as np
import math
# List to store data for the DataFrame
results = []

# for dataset_name, date_tag in dataset_info.items():
for dataset_name in CONSTANTS.DATASET_COLUMN.keys():
    for algo in ['RBRICS']:
        date_tag = None # Put 1225 or other customs tags here
        date_tag = f"{algo}{CONSTANTS.CHOSEN_THRESHOLD[algo][dataset_name]}" if date_tag is None else date_tag
        for fold in [0, 1, 2, 3, 4]:
            lookup, motif_list, motif_counts, motif_lengths, motif_class_count, graph_to_motifs, test_data_lookup, test_graph_to_motifs, train_mask_data, val_mask_data, test_mask_data = get_setup_files_with_folds(dataset_name, date_tag, fold, algo)
            
            # Compute values
            total_train_val_graphs = len(lookup)
            vocab_size = len(motif_list)
            total_test_graphs = len(test_data_lookup)
            graphs_with_motifs_train_val = len(graph_to_motifs.keys())
            total_unique_motifs_train_val = sum([val for values in graph_to_motifs.values() for val in values])
            graphs_with_motifs_test = len(test_graph_to_motifs.keys())
            total_unique_motifs_test = sum([val for values in test_graph_to_motifs.values() for val in values])
            highest_vocab_size = len(motif_counts)
            # Calculate mean
            values = [motif_lengths[key] for key in motif_list if key in motif_lengths]
            mean_motif_length = sum(values) / len(values)

            # Calculate standard deviation
            variance = sum((x - mean_motif_length) ** 2 for x in values) / len(values)
            std_dev_length = math.sqrt(variance)
            
            # Calculate mean
            values = [motif_counts[key] for key in motif_list if key in motif_counts]
            mean_motif_freq = sum(values) / len(values)

            # Calculate standard deviation
            variance = sum((x - mean_motif_freq) ** 2 for x in values) / len(values)
            std_dev_motif_freq = math.sqrt(variance)

            all_possible_occurrences = sum([values for values in motif_counts.values()])
            
            # Add data to results
            results.append({
                'Dataset': dataset_name,
                'DateTag': date_tag,
                'Algorithm': algo,
                'Fold': fold,
                'TotalGraphs_TrainVal': total_train_val_graphs,
                'VocabSize': vocab_size,
                'TotalGraphs_Test': total_test_graphs,
                'GraphsWithMotifs_TrainVal': graphs_with_motifs_train_val,
                'TotalUniqueMotifs_TrainVal': total_unique_motifs_train_val,
                'GraphsWithMotifs_Test': graphs_with_motifs_test,
                'TotalUniqueMotifs_Test': total_unique_motifs_test,
                'HighestVocabSize': highest_vocab_size,
                'AllPossibleOccurrences': all_possible_occurrences,
                'Mean Length': mean_motif_length,
                'Std length':std_dev_length,
                'Mean Frequency': mean_motif_freq,
                'Std Frequency':std_dev_motif_freq,
            })

# Create a DataFrame from the results
df = pd.DataFrame(results)

# # Save to a CSV file (optional)
# df.to_csv('dataset_statistics.csv', index=False)

# Display the DataFrame
print(df)


              Dataset    DateTag Algorithm  Fold  TotalGraphs_TrainVal   
0        Mutagenicity  RBRICS0.2    RBRICS     0                  6904  \
1        Mutagenicity  RBRICS0.2    RBRICS     1                  6904   
2        Mutagenicity  RBRICS0.2    RBRICS     2                  6904   
3        Mutagenicity  RBRICS0.2    RBRICS     3                  6904   
4        Mutagenicity  RBRICS0.2    RBRICS     4                  6904   
5                hERG  RBRICS0.5    RBRICS     0                  8887   
6                hERG  RBRICS0.5    RBRICS     1                  8887   
7                hERG  RBRICS0.5    RBRICS     2                  8887   
8                hERG  RBRICS0.5    RBRICS     3                  8887   
9                hERG  RBRICS0.5    RBRICS     4                  8887   
10               BBBP  RBRICS0.6    RBRICS     0                  1672   
11               BBBP  RBRICS0.6    RBRICS     1                  1672   
12               BBBP  RBRICS0.6    RB

In [ ]:
input()# root_dirs_dict = {"esol":["../1225esol","../1225vanillaregression"],
#                   "hERG":["/nfs/stak/users/kokatea/hpc-share/ChemIntuit/Cluster_JOBS/Regression/KDD/hERG",
#                          "/nfs/stak/users/kokatea/hpc-share/ChemIntuit/Cluster_JOBS/Regression/KDD/Vanilla/85"],
#                   "Lipophilicity":["../1225Lipo","../1225vanillaregression"],
#                   "BBBP":["/nfs/stak/users/kokatea/hpc-share/ChemIntuit/Cluster_JOBS/Regression/KDD/BBBP",
#                              "/nfs/stak/users/kokatea/hpc-share/ChemIntuit/Cluster_JOBS/Regression/KDD/Vanilla/85"],
#                   "Mutagenicity":["/nfs/stak/users/kokatea/hpc-share/ChemIntuit/Cluster_JOBS/Regression/KDD/Mutagenicity",
#                              "/nfs/stak/users/kokatea/hpc-share/ChemIntuit/Cluster_JOBS/Regression/KDD/Vanilla/85"],
#                   "tox21":["../0201","../0201rest",
#                              "../0201Vanilla"]
#                  }

In [ ]:
df.columns

In [4]:
# Select relevant columns
selected_columns = ['Dataset', 'Algorithm', 'Fold', 'VocabSize', 'Mean Length', 'Std length', 'Mean Frequency', 'Std Frequency','TotalGraphs_TrainVal','GraphsWithMotifs_TrainVal']
df_selected = df[selected_columns]

# Group by 'Dataset_Fold' and calculate mean
grouped = df_selected.groupby(['Dataset','Algorithm']).agg({'VocabSize':'mean','Mean Length': 'mean', 'Mean Frequency': 'mean','TotalGraphs_TrainVal':'mean','GraphsWithMotifs_TrainVal':'mean'})

grouped['Graph Coverage'] = grouped['GraphsWithMotifs_TrainVal']/grouped['TotalGraphs_TrainVal']
# Reset index to remove Dataset_Fold column
result = grouped.reset_index()

print(result)

             Dataset Algorithm  VocabSize  Mean Length  Mean Frequency   
0    Alkane_Carbonyl    RBRICS      154.8     4.688100      114.798903  \
1               BBBP    RBRICS      222.8     5.485442       35.977598   
2            Benzene    RBRICS      116.4     4.262807      383.963798   
3  Fluoride_Carbonyl    RBRICS      198.4     4.866688      156.509785   
4      Lipophilicity    RBRICS      148.4     5.372962      137.384940   
5       Mutagenicity    RBRICS      190.0     4.730030       96.680008   
6               esol    RBRICS      174.2     5.175698       13.659506   
7               hERG    RBRICS      180.2     5.064861      284.470870   
8              tox21    RBRICS      219.0     4.466679      118.005645   

   TotalGraphs_TrainVal  GraphsWithMotifs_TrainVal  Graph Coverage  
0                3892.0                     3764.6        0.967266  
1                1672.0                     1565.2        0.936124  
2               10783.2                    10354.4  

In [ ]:
result.to_latex("vocab_info_0205.tex")

In [ ]:
result.to_csv('vocab_statistics_grouped.csv', index=False)

In [ ]:
df.to_csv('dataset_statistics.csv', index=False)